In [ ]:
import pandas as pd

In [ ]:
%pip install xlrd

In [ ]:
'''
At this point you should paste the file attached in this repository under the name GSAF5.xls
'''

file_path = r'c:\Users\llam_\bootcamp\projecte\Proyecto-Shark-Attacks\DateSet_Ataque_Tiburones.xls'
shark_df = pd.read_excel(file_path)

In [ ]:
shark_df.shape

In [ ]:
shark_df.head()

In [ ]:
''' 
We normalize the names of the columns in the dataset and keep the colums with valuable infomration to our proyect.
'''
shark_df.columns = shark_df.columns.str.lower().str.strip().str.replace(' ','_')
cols_to_keep = ['date', 'year', 'country', 'state', 'location', 'species']
shark_df = shark_df[cols_to_keep].copy()
shark_df.head()

In [ ]:
'''
Since the dataset covers an extensive period, we have decided to keep only the records from the last 26 years.
(roughly the average lifespan of a shark)
'''

def filter_years(df):
    """
    Filters the dataframe to keep only rows where the year 
    is between 2000 and 2026 (inclusive).
    """
    # We define the condition: year must be >= 2021 AND <= 2026
    df = df[(df['year'] >= 2000) & (df['year'] <= 2026)]
    
    # It is a good practice to reset the index after filtering rows
    return df.reset_index(drop=True)
# Apply the function
shark_df = filter_years(shark_df)


In [ ]:
shark_df['year'] = shark_df['year'].astype('Int64') 
shark_df.head()

In [ ]:
'''
Knowing the exact location where the event was reported is important for our implementation. 
We only want to keep data in the 'country' column if we can link it to a specific state and location."
'''
shark_df['country'] = shark_df['country'].str.lower().str.strip()
shark_df = shark_df[shark_df['state'].notnull() & shark_df['location'].notnull()]

In [ ]:
shark_df.shape

In [ ]:
shark_df.isnull().sum()

In [ ]:
shark_df['species'] = shark_df['species'].fillna('unknown').str.lower().str.strip()
contains_shark = shark_df['species'].str.contains('shark', na = False)
contains_question = shark_df['species'].str.contains(r'\?|female|not|involvement|or|possibly|juvenile', na = False)
shark_df.loc[~contains_shark | contains_question , 'species'] = 'unknown'
regex_limpieza = r'\d+\s*[m\']|[0-9,\'\.\[\]\"\(\)]|\bto\b|\ba\s+|\bfemale\b|\bmale\b|\bft\b'
regex_estetica = r'\s+'
shark_df['species'] = (shark_df['species']
    .str.replace(regex_limpieza, '', regex=True)
    .str.replace(regex_estetica, ' ', regex=True)
    .str.strip())

shark_df['species'].value_counts().head(20)

In [ ]:
shark_df['country'].value_counts().head(20)

In [ ]:
shark_df['state'].value_counts().head(30)

In [ ]:
shark_df['location'].value_counts().head(30)
                                        

In [ ]:
import re  

def extract_month_text(df):
    """
    Uses Regex to extract only the month names from the date column.
    """
    # Standardize to string and lowercase
    df['date'] = df['date'].astype(str).str.lower()

    def get_only_words(text):
        # We call the 're' module here
        words = re.findall(r'[a-z]+', text)
        
        # Filter strings shorter than 3 chars (like 'st', 'nd', 'rd')
        clean_words = [w for w in words if len(w) > 2]
        
        return clean_words[0] if clean_words else "unknown"

    # Apply and create the new column
    df['month'] = df['date'].apply(get_only_words)
    return df

In [ ]:
shark_df = extract_month_text(shark_df)
shark_df.head()

In [ ]:
shark_df['month'].value_counts()

In [ ]:
def normalize_months(df):

    month_map = {
        'jan': 'january',
        'feb': 'february',
        'mar': 'march',
        'apr': 'april',
        'may': 'may',
        'jun': 'june',
        'jul': 'july',
        'aug': 'august',
        'sep': 'september',
        'oct': 'october',
        'nov': 'november',
        'dec': 'december'
    }
    
    
    df['month'] = df['month'].replace(month_map)
    
    return df

shark_df = normalize_months(shark_df)

shark_df[['month']].value_counts()

In [ ]:
print(shark_df.columns)

In [ ]:
# 1. Standardize text to ensure a perfect match (lowercase and remove spaces)
shark_df['species'] = shark_df['species'].astype(str).str.lower().str.strip()

# 2. Define the list of values to be removed
values_to_remove = ['unknown', 'shark', 'small shark', 'kg shark']

# 3. Filter the dataframe: keep only rows where species is NOT in the list
# The '~' symbol means "NOT"
shark_df = shark_df[~shark_df['species'].isin(values_to_remove)]

print(f"New dataframe shape: {shark_df.shape}")
print(f"Remaining unique values in 'species': {shark_df['species'].nunique()}")

In [ ]:
shark_df['species'].value_counts()

In [ ]:

shark_df.duplicated().sum()

In [ ]:
shark_df = shark_df.drop_duplicates()


In [ ]:
# 1. Define the list of valid months we want to keep
valid_months = [
    'january', 'february', 'march', 'april', 'may', 'june', 
    'july', 'august', 'september', 'october', 'november', 'december'
]

# 2. Filter the dataframe to keep only rows where 'month' is in our list
# This automatically removes 'reported', 'unknown', 'late', 'fall', 'summer', etc.
shark_df = shark_df[shark_df['month'].isin(valid_months)]
print(shark_df['month'].value_counts())

In [ ]:
# 1. Grouping by Country, Month, and Species
# I have removed the extra space after 'species' to match your Index
attacks_detailed = shark_df.groupby(['country', 'month', 'species']).size().reset_index(name='AttackCount')

# 2. Sorting the results
# Sorting alphabetically by country, and then by attack volume
attacks_detailed = attacks_detailed.sort_values(by=['country', 'AttackCount'], ascending=[True, False])

# 3. Final display
print("Detailed Shark Attacks (Cleaned Columns):")
display(attacks_detailed.head(50))

In [ ]:
stats = shark_df[['country', 'month', 'species']].describe()

display(stats)

In [ ]:
# 1. Create a summary table for categorical columns
# We select the columns of interest
top_stats = shark_df[['country', 'month', 'species']].describe().loc[['top', 'freq']]

# 2. Add a 'percentage' row to see how much the top value represents over the total
# (This gives you more context: Is it a clear winner or just by a little bit?)
total_rows = len(shark_df)
top_stats.loc['% of total'] = (top_stats.loc['freq'] / total_rows * 100).round(2).astype(str) + '%'

# 3. Transpose it (T) to make it look like a nice vertical table
summary_table = top_stats.T

print("Shark Attack Summary Table:")
display(summary_table)

In [ ]:
# Definir les columnes que volem analitzar
columns_to_analyze = ['country', 'month', 'species']
total_records = len(shark_df)

for col in columns_to_analyze:
    print(f"\n--- TOP 5 {col.upper()} ---")
    
    # 1. Calcular el recompte de valors (Top 5)
    counts = shark_df[col].value_counts().head(5)
    
    # 2. Calcular el percentatge
    percentages = (shark_df[col].value_counts(normalize=True).head(5) * 100).round(2)
    
    # 3. Combinar-ho en un DataFrame per mostrar-ho bonic
    top_5_df = pd.DataFrame({
        'Total Attacks': counts,
        'Percentage (%)': percentages.astype(str) + '%'
    })
    
    display(top_5_df)

In [ ]:
# 1. Get the Top 5 countries with most attacks
top_5_countries = shark_df['country'].value_counts().head(5).index.tolist()

# 2. Filter the dataframe to include only these top 5 countries
top_df = shark_df[shark_df['country'].isin(top_5_countries)]

# 3. Group by country, month, and species to find the most common patterns
# We calculate the size of each group
top_patterns = top_df.groupby(['country', 'month', 'species']).size().reset_index(name='attack_count')

# 4. Sort and get the top result for each country
# We sort by country and attack_count (descending)
top_patterns = top_patterns.sort_values(['country', 'attack_count'], ascending=[True, False])

# 5. Keep only the #1 most frequent combination for each of the top 5 countries
final_top_stats = top_patterns.groupby('country').head(1)

print("The most frequent Month and Species for the Top 5 Countries:")
display(final_top_stats)


In [ ]:
# 1. Grouping by Country, State, and Location
# We count attacks and find the most frequent species for each spot
detailed_locations = shark_df.groupby(['country', 'state', 'location']).agg(
    total_attacks=('species', 'count'),
    most_frequent_species=('species', lambda x: x.value_counts().index[0] if not x.empty else 'unknown')
).reset_index()

# 2. Sorting by the places with the most attacks
detailed_locations = detailed_locations.sort_values(by='total_attacks', ascending=False)

# 3. Displaying the Top 10 "Hotspots" in the world
print("The 10 locations worldwide with the highest shark attack concentration:")
display(detailed_locations.head(10))

In [ ]:
def get_country_top_analysis(df, country_name):
    # 1. Filtramos el país en minúscula
    country_df = df[df['country'] == country_name.lower()]
    
    # 2. Cogemos los 5 estados con más ataques
    top_states = country_df['state'].value_counts().head(5).index
    
    # 3. Filtrem el dataframe sólo con estos 5 estados
    final_df = country_df[country_df['state'].isin(top_states)]
    
    # 4. Agrupamos per State, Location y Species
    analysis = final_df.groupby(['state', 'location', 'species']).size().reset_index(name='attack_count')
    
    # 5. Ordenamos por estado y número de ataques
    analysis = analysis.sort_values(by=['state', 'attack_count'], ascending=[True, False])
    
    # 6. Para cada estado, nos quedamos con su "Top" (para no hacer la tabla infinita)
    # Mostraremos las 5 localizaciones/especies con más peso de cada estado top
    result = analysis.groupby('state').head(5)
    
    return result

In [ ]:
print("--- TOP 5 ANALYSIS: USA ---")
usa_table = get_country_top_analysis(shark_df, 'usa')
display(usa_table)

In [ ]:
print("--- TOP 5 ANALYSIS: AUSTRALIA ---")
aus_table = get_country_top_analysis(shark_df, 'australia')
display(aus_table)

In [ ]:
print("--- TOP 5 ANALYSIS: SOUTH AFRICA ---")
sa_table = get_country_top_analysis(shark_df, 'south africa')
display(sa_table)

In [ ]:
print("--- TOP 5 ANALYSIS: NEW ZEALAND ---")
nz_table = get_country_top_analysis(shark_df, 'new zealand')
display(nz_table)

In [ ]:
print("--- TOP 5 ANALYSIS: BAHAMAS ---")
bah_table = get_country_top_analysis(shark_df, 'bahamas')
display(bah_table)

In [ ]:
# 1. Definimos la lista de tiburones
available_species = [
    'white shark', 'bull shark', 'tiger shark', 'blacktip shark', 
    'bronze whaler shark', 'nurse shark', 'wobbegong shark', 'mako shark', 
    'lemon shark', 'raggedtooth shark', 'sandtiger shark', 
    'oceanic whitetip shark', 'reef shark', 'blue shark', 
    'grey reef shark', 'spinner shark', 'great white shark'
]

def shark_locator(df):
    print("--- WELCOME TO THE SHARK LOCATOR ---")
    print(f"Available species: {', '.join(available_species)}")
    print("-" * 40)
    
    # Pedimos el input al usuario
    choice = input("Which shark are you looking for? ").lower().strip()
    
    if choice not in available_species:
        print(f"Sorry, '{choice}' is not in our top records. Please check the spelling.")
        return

    # 2. Filtramos el Df por la espécie escogida
    species_df = df[df['species'] == choice]
    
    if species_df.empty:
        print(f"No records found for {choice} after the cleaning process.")
        return

    # 3. Buscamos la localización con más ataques
    # Agrupamos, país, estado y localización
    best_spot = species_df.groupby(['country', 'state', 'location']).size().reset_index(name='count')
    best_spot = best_spot.sort_values('count', ascending=False).iloc[0]
    
    # 4. Resultat
    print(f"\nSTATISTICAL RESULT FOR: {choice.upper()}")
    print(f"If you want to see this shark, the place with the most records is:")
    print(f"📍 Country: {best_spot['country'].upper()}")
    print(f"📍 State:   {best_spot['state'].title()}")
    print(f"📍 Location: {best_spot['location'].title()}")
    print(f"Total historical records in this spot: {best_spot['count']}")

# LLamamos a la función
shark_locator(shark_df)